In [1]:
# 04_cross_country.ipynb
# Cross-national replication of Module 1 (signal asymmetry) and Module 2
# (baseline-sensitivity correlation) across eight education systems.
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
# --- grayscale publication style ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,
                     "axes.titlesize":1,"figure.autolayout":True})
GRAY = ["0.20","0.45","0.65","0.80"]
# --- resolve folders relative to repo (no path is printed, for anonymity) ---
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    # save both formats at 600 dpi, no caption/title; paths are never printed
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)

RES_S=["C_ICTDEV","C_ICTSTD","C_RATDEV","C_RATSTD","C_RATSMB","C_ICTRES","C_RESTOOL","C_RESUTIL","C_HINRES"]
PART=["S_EXCOMP","S_ICTSTUD","S_LRNINTS","S_LRNINTO","S_ACMULT","S_GENEFF","S_SPECEFF","S_CTCLS",
      "S_LRNSAFE","S_GENCLASS","S_SPECLASS","S_ICTPOSG","S_ICTFUT","S_ICTPOSS"]
LEVERS=["S_EXCOMP","S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]
CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT",
     "S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]
COUNTRIES=["kor","deu","dnk","fin","aut","ita","prt","fra"]
def clean(df):
    for c in set(PART+CTX+["NISB"]):
        if c in df: df.loc[(df[c]>=995)|(df[c]<=-995),c]=np.nan
    df.loc[(df.NISB<-6)|(df.NISB>6),"NISB"]=np.nan; return df
def oos(df,cols):
    X=df[cols].to_numpy(float); w=df.TOTWGTS.to_numpy(); g=df.IDSCHOOL.to_numpy(); gkf=GroupKFold(4); s=[]
    for pv in PVS:
        y=df[pv].to_numpy(float); f=[]
        for tr,te in gkf.split(X,y,groups=g):
            m=HistGradientBoostingRegressor(max_iter=250,learning_rate=0.05,max_leaf_nodes=31,
                l2_regularization=1.0,random_state=0).fit(X[tr],y[tr],sample_weight=w[tr])
            f.append(wR2(y[te],m.predict(X[te]),w[te]))
        s.append(np.mean(f))
    return np.mean(s)
rows=[]
for cc in COUNTRIES:
    stu,_=pyreadstat.read_sas7bdat(str(DATA/("bsg"+cc+"i3.sas7bdat")))
    sch,_=pyreadstat.read_sas7bdat(str(DATA/("bcg"+cc+"i3.sas7bdat")))
    df=stu.merge(sch[["IDCNTRY","IDSCHOOL"]+[c for c in RES_S if c in sch]],on=["IDCNTRY","IDSCHOOL"],how="left")
    df=clean(df); df=df[df.TOTWGTS.notna()&df.IDSCHOOL.notna()].copy(); w=df.TOTWGTS.to_numpy()
    r_res=oos(df,[c for c in RES_S if c in df]+["S_INTNET"]); r_par=oos(df,PART)
    x=df.NISB.to_numpy(); ok=~np.isnan(x); o=np.argsort(x[ok]); cw=np.cumsum(w[ok][o])/w[ok].sum(); q=np.interp([1/3,2/3],cw,x[ok][o])
    df["SES3"]=pd.Series(np.where(x<=q[0],"Low",np.where(x<=q[1],"Mid","High"))).where(ok).values
    df["SEX"]=df.S_SEX.map({0.0:"Sex0",1.0:"Sex1"}); d2=df.dropna(subset=["SES3","SEX"]).copy()
    W=d2.TOTWGTS.to_numpy(); Xb=d2[LEVERS+CTX].to_numpy(float); Xc=Xb.copy()
    for j,L in enumerate(LEVERS):
        v=d2[L].to_numpy(); mm=~np.isnan(v); vs=np.sort(v[mm]); ww=W[mm][np.argsort(v[mm])]
        tgt=np.interp(0.75,np.cumsum(ww)/ww.sum(),vs); col=Xc[:,j]; Xc[:,j]=np.where(np.isnan(col),col,np.maximum(col,tgt))
    dd=np.zeros((len(d2),5))
    for k,pv in enumerate(PVS):
        m=HistGradientBoostingRegressor(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,
            l2_regularization=1.0,random_state=0).fit(Xb,d2[pv].to_numpy(float),sample_weight=W)
        dd[:,k]=m.predict(Xc)-m.predict(Xb)
    d2["dCIL"]=dd.mean(1)
    gb=d2.groupby(["SES3","SEX"]).apply(lambda g:pd.Series(dict(
        base=np.average(g.PV1CIL,weights=g.TOTWGTS),dCIL=np.average(g.dCIL,weights=g.TOTWGTS)))).reset_index()
    immig=100*(df.get("S_IMMBGR",pd.Series([np.nan])).eq(1).mean())
    rows.append(dict(cc=cc.upper(),N=len(df),immig_pct=round(immig,1),R2_res=round(r_res,3),
        R2_part=round(r_par,3),corr_base_sens=round(np.corrcoef(gb.base,gb.dCIL)[0,1],2)))
C=pd.DataFrame(rows); C.to_csv(TAB/"cross_country.csv",index=False)
print(C.to_string(index=False))
# figure: grayscale horizontal bars of baseline-sensitivity correlation by country (no title/caption)
fig,ax=plt.subplots(figsize=(4.2,3.4)); C2=C.sort_values("corr_base_sens")
ax.barh(C2.cc,C2.corr_base_sens,color=["0.3" if v<0 else "0.7" for v in C2.corr_base_sens],edgecolor="black",linewidth=0.6)
ax.axvline(0,color="black",lw=0.8); ax.set_xlabel("corr(baseline, sensitivity)"); ax.set_ylabel("")
savefig(fig,"fig_cross_country")


 cc    N  immig_pct  R2_res  R2_part  corr_base_sens
KOR 3723        0.9   0.011    0.263           -0.97
DEU 5065       26.1  -0.176    0.285            0.35
DNK 3038       10.5  -0.021    0.279           -0.56
FIN 4249        6.2   0.012    0.408           -0.90
AUT 3448       27.0   0.031    0.235           -0.71
ITA 3376       13.6  -0.080    0.261           -1.00
PRT 3650       14.4  -0.174    0.270           -0.92
FRA 3694       16.5  -0.084    0.274           -0.96
